
# Estruturação de camada Bronze
  
* **Objetivo:**
Estrutrar layer Bronze a fim de obter dados brutos sem necessidade de download recorrente do S3
* **Tabelas a serem criadas:**    
  -  order_df
  -  user_df
  -  restaurant_df
  -  test_df


## 1.0 Import & Parameters

### 1.1 Imports

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
from pyspark import SparkContext
from pyspark import StorageLevel
import math
from datetime import datetime, date, timedelta
from dateutil.relativedelta import relativedelta
from itertools import combinations
import requests
import gzip
import tarfile
import io
import pandas as pd
from functools import reduce
import os
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from ifood_databricks.toolbelt.geo import get_geodesic_distance

import html           
import unicodedata
import re

from ifood_databricks.toolbelt.data_quality_validator import ifoodDataQualityValidator as dqv
from ifood_databricks import datalake, etl

import numpy as np
from scipy import stats
from scipy.stats import t, mannwhitneyu, ttest_ind

## 1.2 Parameters

### 1.2.1 URL`s

In [0]:
URLS = {
    "order":      "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/order.json.gz",
    "user":       "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/consumer.csv.gz",
    "restaurant": "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/restaurant.csv.gz",
    "ab_test":    "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/ab_test_ref.tar.gz"
    }

### 1.2.2 Read

In [0]:
#### Função para leitura de arquivos

def load_dataset(url: str):
    print(f"\n Lendo: {url}")

    u = url.lower()

    if u.endswith((".json.gz", ".csv.gz")):
        s3_url = (
            url.replace("https://", "s3://")
               .replace(".s3-sa-east-1.amazonaws.com", ""))

        print(f" arquivo: {s3_url}")
        if s3_url.endswith((".json.gz")):
            return spark.read.json(s3_url)
        if s3_url.endswith((".csv.gz")):
            return spark.read.option("header", True).csv(s3_url)
        
    # 2- TAR.GZ
    if u.endswith((".tar.gz")):
        print(" baixando arquivo .tar.gz")
        r = requests.get(url)
        r.raise_for_status()

        with tarfile.open(fileobj=io.BytesIO(r.content), mode="r:gz") as tar:
            target = next(
                (m for m in tar.getmembers()
                 if m.name.endswith(".csv") and not m.name.startswith("._")),
                None)
            if not target:
                raise FileNotFoundError("CSV não é válido dentro de TAR.GZ.")
              
            print(f" CSV Interno: {target.name}")
            f = tar.extractfile(target)
            try:
                pdf = pd.read_csv(f)
            except:
                f.seek(0)
                pdf = pd.read_csv(f, encoding="latin-1")

        return spark.createDataFrame(pdf)

    raise Exception(f"Formato não suportado: {url}")


## 2. Dataset

### 2.1 Loading dataset

In [0]:
## Utilização de função para criaçãod de DF
order_df = load_dataset(URLS["order"])
user_df = load_dataset(URLS["user"])
restaurant_df = load_dataset(URLS["restaurant"])
test_df = load_dataset(URLS["ab_test"])


 Lendo: https://data-architect-test-source.s3-sa-east-1.amazonaws.com/order.json.gz
 arquivo: s3://data-architect-test-source/order.json.gz

 Lendo: https://data-architect-test-source.s3-sa-east-1.amazonaws.com/consumer.csv.gz
 arquivo: s3://data-architect-test-source/consumer.csv.gz

 Lendo: https://data-architect-test-source.s3-sa-east-1.amazonaws.com/restaurant.csv.gz
 arquivo: s3://data-architect-test-source/restaurant.csv.gz

 Lendo: https://data-architect-test-source.s3-sa-east-1.amazonaws.com/ab_test_ref.tar.gz
 baixando arquivo .tar.gz
 CSV Interno: ab_test_ref.csv



# Estruturação de camada Silver
  
* **Objetivo:**
Estrutrar layer Silver a fim de tratar dados e adicionar dados para análises mais detalhadas
* **Ações a serem feitas:**    
  -  Tratamento de dados
  -  Input de informações relevantes
  -  Validadores de dados
  -  União de tabelas para criação de tabela única


## 3.0 Transforming

### 3.1 Functions

In [0]:
### Função para tratamento de dados String 

def normalize_str(column_name):
    
       # Cast inicial para string
    col_ref = col(column_name).cast(StringType())
    
    # Aplicar transformações em sequência
    result = col_ref
    
    # 1. Trim inicial e normalizar espaços múltiplos
    result = trim(regexp_replace(result, "\\s+", " "))
    
    # 2. Conversão para maiúscula
    result = upper(result)
    
    # 3. Remoção de acentos (mapeamento completo)
    accented_chars = "áàâãäÁÀÂÃÄéèêëÉÈÊËíìîïÍÌÎÏóòôõöÓÒÔÕÖúùûüÚÙÛÜçÇñÑ"
    clean_chars = "aaaaaAAAAAeeeeEEEEiiiiIIIIoooooOOOOOuuuuUUUUcCnN"
    result = translate(result, accented_chars, clean_chars)
    
    # 4. Manter apenas alfanuméricos, números e espaços
    result = regexp_replace(result, "[^A-Z0-9 ]", "")
    
    # 5. Trim final e limpeza de espaços duplos
    result = trim(regexp_replace(result, "\\s+", " "))
    
    # 6. Tratamento robusto de nulls e strings vazias
    return when(
        col_ref.isNull() | 
        (length(trim(coalesce(col_ref, lit("")))) == 0) | 
        (result == "") |
        result.isNull(),
        None
    ).otherwise(result)


In [0]:
### Tratamento de Coluna item na orders_df
item_schema = ArrayType(
    StructType([
        StructField("name", StringType(), True),
        StructField("addition", StructType([
            StructField("value", DoubleType(), True),
            StructField("currency", StringType(), True)
        ]), True),
        StructField("discount", StructType([
            StructField("value", DoubleType(), True),
            StructField("currency", StringType(), True)
        ]), True),
        StructField("quantity", DoubleType(), True),
        StructField("sequence", IntegerType(), True),
        StructField("unitPrice", StructType([
            StructField("value", DoubleType(), True),
            StructField("currency", StringType(), True)
        ]), True),
        StructField("externalId", StringType(), True),
        StructField("totalValue", StructType([
            StructField("value", DoubleType(), True),
            StructField("currency", StringType(), True)
        ]), True),
        StructField("customerNote", StringType(), True),
        StructField("garnishItems", ArrayType(StructType([
            StructField("name", StringType(), True),
            StructField("addition", StructType([
                StructField("value", DoubleType(), True),
                StructField("currency", StringType(), True)
            ]), True),
            StructField("discount", StructType([
                StructField("value", DoubleType(), True),
                StructField("currency", StringType(), True)
            ]), True),
            StructField("quantity", DoubleType(), True),
            StructField("sequence", IntegerType(), True),
            StructField("unitPrice", StructType([
                StructField("value", DoubleType(), True),
                StructField("currency", StringType(), True)
            ]), True),
            StructField("categoryId", StringType(), True),
            StructField("externalId", StringType(), True),
            StructField("totalValue", StructType([
                StructField("value", DoubleType(), True),
                StructField("currency", StringType(), True)
            ]), True),
            StructField("categoryName", StringType(), True),
            StructField("integrationId", StringType(), True)
        ])), True),
        StructField("integrationId", StringType(), True),
        StructField("totalAddition", StructType([
            StructField("value", DoubleType(), True),
            StructField("currency", StringType(), True)
        ]), True),
        StructField("totalDiscount", StructType([
            StructField("value", DoubleType(), True),
            StructField("currency", StringType(), True)
        ]), True)
    ])
)

### 3.2 Data Treatment 

In [0]:
## Seleção de colunas buscando anonimização de dados, tratamento do tipo de dado e redundancia

order_df = order_df.select(
  
  normalize_str('order_id').alias('order_id'),
  col('order_created_at').cast(TimestampType()),
   
  normalize_str('customer_id').alias('customer_id'),
  
  col('order_scheduled').cast(BooleanType()),
  col('order_scheduled_date').cast(TimestampType()),
  
  normalize_str('merchant_id').alias('merchant_id'),

  
  col('order_total_amount').cast(DoubleType()),
  from_json(col("items"), item_schema).alias("items"),  

  
  col('merchant_latitude').cast(DoubleType()),
  col('merchant_longitude').cast(DoubleType()),
  normalize_str('delivery_address_district').alias('delivery_address_district'), 
  normalize_str('delivery_address_city').alias('delivery_address_city'), 
  normalize_str('delivery_address_state').alias('delivery_address_state'),
  normalize_str('delivery_address_country').alias('delivery_address_country'), 
  col('delivery_address_latitude').cast(DoubleType()),
  col('delivery_address_longitude').cast(DoubleType()),
  normalize_str('delivery_address_zip_code').alias('delivery_address_zip_code'), 
  normalize_str('delivery_address_external_id').alias('delivery_address_external_id'),

  normalize_str('origin_platform').alias('origin_platform'), 

  normalize_str('merchant_timezone').alias('merchant_timezone')
).dropDuplicates()

In [0]:
user_df = user_df.select(
  normalize_str('customer_id').alias('customer_id'),

  col('active').cast(BooleanType()).alias('active_user'),
  col('created_at').cast(TimestampType()).alias('user_created_at'),
  normalize_str('language').alias('language') 
).dropDuplicates()


In [0]:
restaurant_df = restaurant_df.select(
  normalize_str('id').alias('merchant_id'),

  
  col('created_at').cast(TimestampType()).alias('merchant_created_at'),

  normalize_str('merchant_city').alias('merchant_city'),
  normalize_str('merchant_state').alias('merchant_state'),
  normalize_str('merchant_country').alias('merchant_country'),
  normalize_str('merchant_zip_code').alias('merchant_zip_code'), 

  
  col('minimum_order_value').cast(DoubleType()), 
  col('average_ticket').cast(DoubleType()), 

  
  col('price_range').cast(IntegerType()), 
  col('takeout_time').cast(IntegerType()),
  col('delivery_time').cast(IntegerType()),

  
  col('enabled').cast(BooleanType()).alias('enabled_restaurant')

).dropDuplicates()


In [0]:
test_df = test_df.select(
  
  normalize_str('customer_id').alias('customer_id'),
  normalize_str('is_target').alias('is_target')
  ).filter(col('customer_id').isNotNull()
           ).dropDuplicates()


### 3.3 Data input 

#### 3.3.1 Distancia entre merchant e entrega

In [0]:
## Criação de informação referente a raio de entrega
order_df = order_df.withColumn('distancy',get_geodesic_distance(
                                                      array('merchant_latitude', 'merchant_longitude'),
                                                      array('delivery_address_latitude', 'delivery_address_longitude')))

In [0]:
display(order_df.agg({'distancy': 'max'}))

# valor em duplicada das coordenadas

In [0]:
# Ajuste de tabela order a fim de retirada de redundancia
order_df = order_df.groupBy(
  
  col('order_id'),
   
  col('customer_id'),

  col('order_scheduled'),
  col('order_scheduled_date'),
  
  col('merchant_id'),

  col('order_total_amount'),
  col('items'),  
  
  col('merchant_latitude'),
  col('merchant_longitude'),
  col('delivery_address_district'), 
  col('delivery_address_city'), 
  col('delivery_address_state'),
  col('delivery_address_country'), 
  col('delivery_address_zip_code'), 
  col('delivery_address_external_id'),

  col('origin_platform'), 

  col('merchant_timezone')
  ).agg(min(col('order_created_at')).alias('order_created_at')  ## Valor em duplicata em algumas orders
        ).filter(col('customer_id').isNotNull()
                 ).filter((col('order_total_amount')> 0.0)&(col('order_total_amount').isNotNull())
                          ).dropDuplicates()


#### 3.3.2 Lifecycle semanal

In [0]:
## Criação de lifecycle semanal
# STEP 1: Agrupar por semana
db_week = order_df.select(
    date_trunc('week', col('order_created_at')).cast('date').alias('week'),
    col('customer_id'),
    col('order_id')
).groupBy('week', 'customer_id').agg(
    countDistinct('order_id').alias('orders_by_week')
).orderBy('week')

# STEP 2: Aplicar LAG
window_spec = Window.partitionBy('customer_id').orderBy('week')

db_week_lag = db_week.select(
    col('week'),
    col('customer_id'),
    col('orders_by_week'),
    lag('orders_by_week', 1).over(window_spec).alias('orders_last_week'),
    lag('week', 1).over(window_spec).alias('previous_order')
).withColumn(
    'week_diff',
    (datediff(col('week'), col('previous_order')) / 7).cast('integer')
)

# STEP 3: NOVA CATEGORIZAÇÃO
lifecycle_df = db_week_lag.select(
    col('week'),
    col('customer_id'),
    col('orders_by_week'),
    col('orders_last_week'),
    col('previous_order'),
    col('week_diff'),
    
    # LÓGICA DE CATEGORIZAÇÃO
    when(col('orders_last_week').isNull(), 'new')
    .when(
        # Continuidade sem gap: hot active vs active
        (col('week_diff') == 1) & (col('orders_by_week') >= 2), 
        'hot active'
    )
    .when(
        (col('week_diff') == 1) & (col('orders_by_week') == 1), 
        'active'
    )
    .when(
        # Diferentes níveis de reativação baseados no gap
        col('week_diff') == 2, 'reactivated'          # 1 semana de gap
    )
    .when(
        col('week_diff') == 3, 'hot reactivated'      # 2 semanas de gap
    )
    .when(
        col('week_diff') == 4, 'warm reactivated'     # 3 semanas de gap  
    )
    .when(
        col('week_diff') >= 5, 'cold reactivated'     # 4+ semanas de gap
    )
    .otherwise('others')
    .alias('lifecycle')
)

In [0]:
### Criação de coluna com inforamção semanal
order_df = order_df.withColumn(
    'week', date_trunc('week', col('order_created_at')).cast('date'))

### 3.4 Data Validator 

In [0]:
## Validador da orders
# dataframe temporario para realizar o dump
validation_order = datalake.dataframe2tempdataset(dataframe = order_df,
                                          namespace='loyalty_clube',
                                           dataset = 'validation_order',
                                            force_s3 = True)

# Configuração do validador de qualidade de dados
validator = dqv(
      validation_order,
      displayName=f"Validação de order_id",
      slackUsername="@jhonathan.silva",
      slack_direct=True,
      dataset=f'validation_order'
)

validator = validator \
    .hasUniqueKey('order_id') \
    .isNeverNull('order_id')

validator.run()

[INFO][2026-01-14 19:56:51,230][datalake.py    ][1023][MainThread] - Writing temporary dataset.
INFO:ifood_databricks.log:Writing temporary dataset.
[INFO][2026-01-14 19:56:51,245][datalake.py    ][ 136][MainThread] - Writing temporary table to get statistics...
INFO:ifood_databricks.log:Writing temporary table to get statistics...
[INFO][2026-01-14 20:18:04,457][datalake.py    ][1031][MainThread] - Temp dataset saved in s3://prd-ifood-data-lake-temp-dataframe/temp_loyalty_clube_validation_order_temp
INFO:ifood_databricks.log:Temp dataset saved in s3://prd-ifood-data-lake-temp-dataframe/temp_loyalty_clube_validation_order_temp



Ignore this message if this is the first time the dataset is written.

Validation metrics will be collected without dataset name.
##### Validation Report - VALIDAÇÃO DE ORDER_ID [2026-01-14 17:18:05] #####
It has a total number of 19 columns and 2427313 rows.
##### Metrics Overhaul Result - 0 total dirty records found #####
 - Column 'order_id' is a unique key 
 - Column order_id is never null. 

##### Quality Dimensions Report - VALIDAÇÃO DE ORDER_ID [2026-01-14 17:18:05] #####
+-------------+----------+
| completeness|uniqueness|
+-------------+----------+
|        100.0|     100.0|
+-------------+----------+


In [0]:
## Validador da user
# dataframe temporario para realizar o dump
validation_user = datalake.dataframe2tempdataset(dataframe = user_df,
                                          namespace='loyalty_clube',
                                           dataset = "validation_user",
                                            force_s3 = True)

# Configuração do validador de qualidade de dados
validator = dqv(
      validation_user,
      displayName=f"Validação de customer_id",
      slackUsername="@jhonathan.silva",
      slack_direct=True,
      dataset=f'validation_user'
)

validator = validator \
    .hasUniqueKey('customer_id') \
    .isNeverNull('customer_id')

validator.run()

[INFO][2026-01-14 20:18:14,830][datalake.py    ][1023][MainThread] - Writing temporary dataset.
INFO:ifood_databricks.log:Writing temporary dataset.
[INFO][2026-01-14 20:18:14,846][datalake.py    ][ 136][MainThread] - Writing temporary table to get statistics...
INFO:ifood_databricks.log:Writing temporary table to get statistics...
[INFO][2026-01-14 20:18:30,788][datalake.py    ][1031][MainThread] - Temp dataset saved in s3://prd-ifood-data-lake-temp-dataframe/temp_loyalty_clube_validation_user_temp
INFO:ifood_databricks.log:Temp dataset saved in s3://prd-ifood-data-lake-temp-dataframe/temp_loyalty_clube_validation_user_temp



Ignore this message if this is the first time the dataset is written.

Validation metrics will be collected without dataset name.
##### Validation Report - VALIDAÇÃO DE CUSTOMER_ID [2026-01-14 17:18:31] #####
It has a total number of 4 columns and 806156 rows.
##### Metrics Overhaul Result - 0 total dirty records found #####
 - Column 'customer_id' is a unique key 
 - Column customer_id is never null. 

##### Quality Dimensions Report - VALIDAÇÃO DE CUSTOMER_ID [2026-01-14 17:18:31] #####
+-------------+----------+
| completeness|uniqueness|
+-------------+----------+
|        100.0|     100.0|
+-------------+----------+


In [0]:
## Validador da restaurant
# dataframe temporario para realizar o dump
validation_restaurant = datalake.dataframe2tempdataset(dataframe = restaurant_df,
                                          namespace='loyalty_clube',
                                           dataset = "validation_restaurant",
                                            force_s3 = True)

# Configuração do validador de qualidade de dados
validator = dqv(
      validation_restaurant,
      displayName=f"Validação de merchant_id",
      slackUsername="@jhonathan.silva",
      slack_direct=True,
      dataset=f'validation_restaurant'
)

validator = validator \
    .hasUniqueKey('merchant_id') \
    .isNeverNull('merchant_id')

validator.run()

[INFO][2026-01-14 20:18:35,703][datalake.py    ][1023][MainThread] - Writing temporary dataset.
INFO:ifood_databricks.log:Writing temporary dataset.
[INFO][2026-01-14 20:18:35,717][datalake.py    ][ 136][MainThread] - Writing temporary table to get statistics...
INFO:ifood_databricks.log:Writing temporary table to get statistics...
[INFO][2026-01-14 20:18:42,165][datalake.py    ][1031][MainThread] - Temp dataset saved in s3://prd-ifood-data-lake-temp-dataframe/temp_loyalty_clube_validation_restaurant_temp
INFO:ifood_databricks.log:Temp dataset saved in s3://prd-ifood-data-lake-temp-dataframe/temp_loyalty_clube_validation_restaurant_temp



Ignore this message if this is the first time the dataset is written.

Validation metrics will be collected without dataset name.
##### Validation Report - VALIDAÇÃO DE MERCHANT_ID [2026-01-14 17:18:42] #####
It has a total number of 12 columns and 7292 rows.
##### Metrics Overhaul Result - 0 total dirty records found #####
 - Column 'merchant_id' is a unique key 
 - Column merchant_id is never null. 

##### Quality Dimensions Report - VALIDAÇÃO DE MERCHANT_ID [2026-01-14 17:18:42] #####
+-------------+----------+
| completeness|uniqueness|
+-------------+----------+
|        100.0|     100.0|
+-------------+----------+


In [0]:
## Validador da user_test
# dataframe temporario para realizar o dump
validation_test = datalake.dataframe2tempdataset(dataframe = test_df,
                                          namespace='loyalty_clube',
                                           dataset = "validation_test",
                                            force_s3 = True)

# Configuração do validador de qualidade de dados
validator = dqv(
      validation_test,
      displayName=f"Validação de customer_id",
      slackUsername="@jhonathan.silva",
      slack_direct=True,
      dataset=f'validation_test'
)

validator = validator \
    .hasUniqueKey('customer_id') \
    .isNeverNull('customer_id')

validator.run()

[INFO][2026-01-15 01:24:17,731][datalake.py    ][1023][MainThread] - Writing temporary dataset.
INFO:ifood_databricks.log:Writing temporary dataset.
[INFO][2026-01-15 01:24:17,748][datalake.py    ][ 136][MainThread] - Writing temporary table to get statistics...
INFO:ifood_databricks.log:Writing temporary table to get statistics...
[INFO][2026-01-15 01:24:44,948][datalake.py    ][1031][MainThread] - Temp dataset saved in s3://prd-ifood-data-lake-temp-dataframe/temp_loyalty_clube_validation_test_temp
INFO:ifood_databricks.log:Temp dataset saved in s3://prd-ifood-data-lake-temp-dataframe/temp_loyalty_clube_validation_test_temp



Ignore this message if this is the first time the dataset is written.

Validation metrics will be collected without dataset name.
##### Validation Report - VALIDAÇÃO DE CUSTOMER_ID [2026-01-14 22:24:45] #####
It has a total number of 2 columns and 806466 rows.
##### Metrics Overhaul Result - 0 total dirty records found #####
 - Column 'customer_id' is a unique key 
 - Column customer_id is never null. 

##### Quality Dimensions Report - VALIDAÇÃO DE CUSTOMER_ID [2026-01-14 22:24:45] #####
+-------------+----------+
| completeness|uniqueness|
+-------------+----------+
|        100.0|     100.0|
+-------------+----------+


### 3.5 Join 

In [0]:
#Join de tabelas a fim de criar tabela única
abtest_df_silver = order_df.join(lifecycle_df, ['customer_id', 'week'], 'inner') \
                            .join(user_df, ['customer_id'], 'left') \
                            .join(test_df, ['customer_id'], 'inner') \
                            .join(restaurant_df, ['merchant_id'], 'left') 



# Estruturação de camada Analítica
  
* **Objetivo:**
Trazer dados de forma mais detalhada, transformando-os em insigths
* **Modulos a serem desenvolvidos:**    
  -  ANÁLISE EXPLORATÓRIA
  -  SEGMENTAÇÃO DE USUÁRIOS
  -  ANÁLISE DE NEGÓCIO E ROI

* **Gráficos com visualizações:**    
  -  https://docs.google.com/spreadsheets/d/1WXukj8eeSnyuGTEe5BPJVzVTJRoOgL8EBKCyIIec3TI/edit?usp=sharing

## 4.0 Big numbers

In [0]:
df_analysis = abtest_df_silver.groupBy('is_target').agg(
    countDistinct('customer_id').alias('usuarios'),
    avg('order_total_amount').alias('aov'),
    (count('*') / countDistinct('customer_id')).alias('freq'),
    (countDistinct(when(col('active_user') == True, col('customer_id'))) / countDistinct('customer_id')).alias('share_active_user'),
    (countDistinct(when(col('enabled_restaurant') == True, col('merchant_id'))) / countDistinct('merchant_id')).alias('share_active_merchant'),


    )

display(df_analysis)

is_target,usuarios,aov,freq,share_active_user,share_merchant_user
TARGET,445909,47.81117965091096,3.176912778167743,0.997616105528258,0.5598450255984503
CONTROL,360528,47.91951262443228,2.803385590023521,0.9976673101673101,0.5614230127848805


## 4.1 Segmentação de usuários

### 4.1.1 Segmentação por lifecycle

In [0]:
### BIG NUMBERS POR LIFECYCLE SEMANAL
df_analysis_lifecycle = abtest_df_silver.groupBy('is_target','week', 'lifecycle').agg(
    countDistinct('customer_id').alias('usuarios'),
    count('order_id').alias('pedidos'))
display(df_analysis_lifecycle)

is_target,week,lifecycle,usuarios,pedidos
CONTROL,2019-01-14,cold reactivated,6611,8074
TARGET,2019-01-21,new,23188,25973
CONTROL,2018-12-10,hot active,12938,35403
CONTROL,2018-12-17,reactivated,10836,15007
TARGET,2018-12-31,reactivated,14370,19569
CONTROL,2018-12-24,active,19525,19525
TARGET,2019-01-07,reactivated,17198,23791
TARGET,2018-12-24,hot active,18588,50956
TARGET,2019-01-28,hot reactivated,6522,7516
TARGET,2019-01-21,cold reactivated,13498,16369


### 4.1.2 Segmentação por AOV

In [0]:
#ESTRUTURAÇÃO DE BIG NUMBER POR LIFECYCLE AOV

df_analysis_aov = abtest_df_silver.withColumn(
  'flag_aov', when(
        # Diferentes níveis de reativação baseados no gap
        col('order_total_amount') <= 15, 'a) <=15'    
    )
    .when(
        col('order_total_amount') <= 30, 'b) <=30'          # 2 semanas de gap
    )
    .when(
        col('order_total_amount') <= 45, 'c) <=45'          # 2 semanas de gap
    )
    .when(
        col('order_total_amount') <= 60, 'd) <=60'          # 2 semanas de gap
    )
    .when(
        col('order_total_amount') > 60, 'e) >60'          # 2 semanas de gap
    )
  )

In [0]:
### BIG NUMBERS POR AOV
df_aov = df_analysis_aov.groupBy('is_target','week', 'flag_aov').agg(
    countDistinct('customer_id').alias('usuarios'),
    count('order_id').alias('pedidos'))

display(df_aov)    

is_target,week,flag_aov,usuarios,pedidos
CONTROL,2018-12-31,a) <=15,4265,4720
TARGET,2018-12-03,c) <=45,37160,42313
CONTROL,2019-01-14,a) <=15,5119,5924
CONTROL,2019-01-07,e) >60,19727,22881
CONTROL,2018-12-24,a) <=15,5759,6695
CONTROL,2018-12-31,e) >60,20473,23441
CONTROL,2019-01-28,e) >60,16249,18059
TARGET,2018-12-10,a) <=15,7241,8121
TARGET,2018-12-17,c) <=45,39661,45403
TARGET,2018-12-31,a) <=15,5844,6498


### 4.1.3 Clusterização para novos testes

In [0]:
user_features = abtest_df_silver.groupBy('customer_id','is_target').agg(
    count('*').alias('pedidos'),
    avg('order_total_amount').alias('aov'),
    datediff(max('order_created_at'), min('order_created_at')).alias('delta_dias'),
    countDistinct('merchant_id').alias('qtd_restaurantes')
)

#### 4.1.1 Features para nova clusterização

In [0]:
feature_cols = [
    'pedidos', 'aov', 
    'delta_dias', 'qtd_restaurantes'
]

In [0]:
#Oportunidade de uso de Kmeans e VectorAssembler, StandardScaler - limitação por DB
# Função para criação de cluster a partir de KPIs

def clustering_sql_avancado(df):
    
    # Score composto
    df_scored = df.withColumn("score_cliente",
        (col("pedidos") * 0.3) +
        (col("aov") * 0.3) + 
        (col("delta_dias") * 0.3) +
        (col("qtd_restaurantes") * 0.1)
    )
    
    # Clusters usando quartis
    df_clustered = df_scored.withColumn("cluster",
        ntile(4).over(Window.orderBy("score_cliente").partitionBy('is_target'))
    )
    
    return df_clustered

In [0]:
user_clusters_sql = clustering_sql_avancado(user_features)


In [0]:
display(user_clusters_sql)

customer_id,is_target,pedidos,aov,delta_dias,qtd_restaurantes,score_cliente,cluster
B47A69799864DFFDA53D95764F146C1DE0B86C478BCF7104E2FBF307EFD19613,TARGET,1,0.01,0,1,0.403,1
F3A979D13628E9B335DF5D807431AE452F4BD27135151492744FB5C41A890B1D,TARGET,1,1.0,0,1,0.7,1
506C8EC47E21126D3898FD081D01663633C7C2BF93C3CF00C158DF32F748A469,TARGET,1,1.0,0,1,0.7,1
2F13665578D64D9BCEE85BE0F5497C6C689AF960BFABEA3A4DAE860241D66E03,TARGET,1,1.0,0,1,0.7,1
8BEC8E743245AD63E1C3F37E784E21B32292BB5EB8AFC9A403E4D9BB5194E8E5,TARGET,1,1.0,0,1,0.7,1
0556EECE10DDDF9A1BB4F85E275BBEB9DAD4B28896889F0B83F8C41BCDD23EF3,TARGET,1,1.0,0,1,0.7,1
074DF67D6D7282FC7D58212EF12F5E33EEFD3E92A4DD983431E60FCB2CCF26FB,TARGET,1,1.5,0,1,0.85,1
53F7DCA4135F70D19EE99758E313F1BE4A6B60BA458078147A3DA3D249CD7AF4,TARGET,1,1.5,0,1,0.85,1
B06C6255E815EEA1E54C2D2CE461460BACC3CD40C55891E012C30DA0A201DE48,TARGET,1,2.0,0,1,0.9999999999999999,1
E271D4C7CEA039DA703CDCD7079961731D06997D044FA7880962BEDE67E69527,TARGET,1,2.5,0,1,1.1500000000000001,1


## 4.2 Dados para cálculo de ROI e Projeções

In [0]:
user_retention = abtest_df_silver.groupBy('is_target').agg(
    countDistinct(when(col('lifecycle') == 'new', col('customer_id'))).alias('usuarios_novos'),
    countDistinct(when(col('lifecycle') != 'new', col('customer_id'))).alias('usuarios_retidos'),
    sum(when(col('lifecycle') == 'new', col('order_total_amount'))).alias('gmv_novos'),
    sum(when(col('lifecycle') != 'new', col('order_total_amount'))).alias('gmv_retidos')
    )
display(user_retention)

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/databricks/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 541, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/databricks/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/databricks/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 564, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


is_target,usuarios_novos,usuarios_retidos,gmv_novos,gmv_retidos
TARGET,445909,234149,2.6321191889998678E7,4.140879455999378E7
CONTROL,360528,156580,2.0756753450000275E7,2.767545003999853E7


In [0]:
user_retention_react = abtest_df_silver.groupBy('is_target').agg(
    (count(when(col('lifecycle') != 'new', col('order_id'))) / 
     countDistinct(when(col('lifecycle') != 'new', col('customer_id')))).alias('freq_retidos')
    )
display(user_retention_react)

is_target,freq_retidos
TARGET,3.6876646921404745
CONTROL,3.6932302976114446
